# Settimana 20 — Fine-tuning: quando, come e quali alternative ci sono

**Obiettivo:** costruire un classificatore di documenti del settore agroalimentare
usando il fine-tuning di `gpt-4o-mini`, confrontarlo con il few-shot prompting,
e decidere qual è l'opzione migliore per il caso d'uso.

**Caso d'uso:** classificare automaticamente i documenti che arrivano a un'azienda
del settore ittico (report di cattura, certificazioni, normativa, fatture...)
per instradarli al reparto corretto.

**Al termine avrai:**
- Un dataset di fine-tuning in formato `.jsonl`
- Un modello con fine-tuning applicato con `gpt-4o-mini`
- Un confronto: fine-tuned vs few-shot vs modello base
- La decisione motivata su cosa usare in produzione

**Costo stimato:**
- Dataset: $0.00 (generiamo esempi sintetici con GPT)
- Fine-tuning (200 esempi × 3 epoche): ~$0.10–0.20
- Valutazione: ~$0.01
- **Totale: ~$0.15–0.25**

## Cella 0 — Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv"])

import os, json, time, random
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Definisci OPENAI_API_KEY nel tuo .env"

client = OpenAI(api_key=OPENAI_API_KEY)

DATA_DIR = Path("ft_data")
DATA_DIR.mkdir(exist_ok=True)

MODELO_BASE = "gpt-4o-mini"
random.seed(42)

print(f"✓ OpenAI SDK pronto")
print(f"✓ Directory dati: {DATA_DIR.resolve()}")

## Cella 1 — Definire il task e le categorie

Prima di costruire il dataset, definisci esattamente cosa classifica il modello
e quale formato deve restituire.

In [ ]:
# Tassonomia di documenti del settore ittico/agroalimentare
CATEGORIAS = {
    "informes_capturas": {
        "descripcion": "Report di catture, quote, volumi di pesca",
        "ejemplos_titulo": [
            "Report catture nasello caladero Atlantico Nord Q3 2025",
            "Quota annuale merluzzo flotta galiziana 2025",
            "Statistiche sbarco porto di Vigo settembre 2025",
        ]
    },
    "certificaciones": {
        "descripcion": "Certificati di qualità, audit, accreditamenti",
        "ejemplos_titulo": [
            "Certificato IFS Food stabilimento di lavorazione Vigo 2025",
            "Audit BRC magazzino frigorifero A Coruña",
            "Rinnovo ISO 22000 conserviera Galizia",
        ]
    },
    "normativa": {
        "descripcion": "Regolamenti, direttive, legislazione applicabile",
        "ejemplos_titulo": [
            "Regolamento UE 2023/1115 due diligence catena di fornitura",
            "Regio Decreto etichettatura prodotti ittici 2024",
            "Direttiva taglie minime cattura specie demersali",
        ]
    },
    "analisis_mercado": {
        "descripcion": "Report prezzi, tendenze di mercato, concorrenza",
        "ejemplos_titulo": [
            "Prezzi mercato ittico Vigo settimana 42 2025",
            "Analisi importazioni pesce congelato Spagna Q4 2025",
            "Report tendenze consumo prodotti ittici Europa 2025",
        ]
    },
    "documentacion_tecnica": {
        "descripcion": "Schede tecniche, specifiche di prodotto, protocolli",
        "ejemplos_titulo": [
            "Scheda tecnica conserva cozze in scabeccio lattina 120g",
            "Protocollo HACCP linea cottura gamberone",
            "Specifiche microbiologiche prodotto fresco refrigerato",
        ]
    },
}

SYSTEM_PROMPT = """Sei un classificatore di documenti per il settore ittico e agroalimentare.
Classifica il titolo del documento in una di queste categorie esatte:
- informes_capturas
- certificaciones
- normativa
- analisis_mercado
- documentacion_tecnica

Rispondi SOLO con il nome della categoria, senza spiegazioni."""

print(f"Tassonomia: {len(CATEGORIAS)} categorie")
for cat, info in CATEGORIAS.items():
    print(f"  · {cat}: {info['descripcion']}")

## Cella 2 — Generare il dataset con distillazione da GPT-4o

Usiamo GPT-4o per generare esempi di addestramento sintetici.
Questo è il pattern di **distillazione**: usi un modello più costoso per creare
il dataset e poi addestri un modello più economico.

In [ ]:
CACHE_DATASET = DATA_DIR / "dataset_raw.json"

def generar_titulos_categoria(categoria: str, descripcion: str, n: int = 25) -> list[str]:
    """Genera titoli di documenti sintetici per una categoria."""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",   # usiamo mini per abbattere il costo della generazione
        messages=[{
            "role": "user",
            "content": (
                f"Genera {n} titoli realistici di documenti della categoria '{categoria}' "
                f"({descripcion}) per un'azienda del settore ittico e agroalimentare galiziano.\n"
                f"I titoli devono essere vari: prodotti, periodi, enti diversi.\n"
                f"Restituisci SOLO i titoli, uno per riga, senza numerazione né spiegazioni."
            )
        }],
        temperature=0.9,
        max_tokens=800,
    )
    lineas = resp.choices[0].message.content.strip().split("\n")
    return [l.strip() for l in lineas if l.strip()][:n]


if CACHE_DATASET.exists():
    dataset_raw = json.loads(CACHE_DATASET.read_text())
    print(f"✓ Dataset caricato dalla cache ({sum(len(v) for v in dataset_raw.values())} esempi)")
else:
    print("Generazione esempi per categoria...")
    dataset_raw = {}
    for cat, info in CATEGORIAS.items():
        titulos = generar_titulos_categoria(cat, info["descripcion"], n=40)
        dataset_raw[cat] = titulos
        print(f"  ✓ {cat}: {len(titulos)} titoli generati")

    CACHE_DATASET.write_text(json.dumps(dataset_raw, ensure_ascii=False, indent=2))
    print(f"\n✓ Dataset salvato in cache")

total = sum(len(v) for v in dataset_raw.values())
print(f"\nTotale: {total} esempi | {total // len(CATEGORIAS)} per categoria (media)")

## Cella 3 — Costruire gli split train/eval e convertire in JSONL

In [ ]:
def construir_mensaje(titulo: str, categoria: str) -> dict:
    """Converte un esempio nel formato di fine-tuning di OpenAI."""
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": f"Classifica: '{titulo}'"},
            {"role": "assistant", "content": categoria},
        ]
    }


# Costruisce tutti gli esempi
todos = []
for cat, titulos in dataset_raw.items():
    for titulo in titulos:
        todos.append((titulo, cat))

random.shuffle(todos)

# Split 80/20 stratificato per categoria
train_ejemplos, eval_ejemplos = [], []
for cat in CATEGORIAS:
    cat_items = [(t, c) for t, c in todos if c == cat]
    n_train   = int(len(cat_items) * 0.8)
    train_ejemplos.extend(cat_items[:n_train])
    eval_ejemplos.extend(cat_items[n_train:])

random.shuffle(train_ejemplos)
random.shuffle(eval_ejemplos)

# Salva i file JSONL
TRAIN_FILE = DATA_DIR / "train.jsonl"
EVAL_FILE  = DATA_DIR / "eval.jsonl"

for ruta, ejemplos in [(TRAIN_FILE, train_ejemplos), (EVAL_FILE, eval_ejemplos)]:
    with ruta.open("w", encoding="utf-8") as f:
        for titulo, cat in ejemplos:
            f.write(json.dumps(construir_mensaje(titulo, cat), ensure_ascii=False) + "\n")

print(f"✓ Train: {len(train_ejemplos)} esempi → {TRAIN_FILE}")
print(f"✓ Eval:  {len(eval_ejemplos)} esempi → {EVAL_FILE}")
print()

# Mostra la distribuzione
from collections import Counter
dist_train = Counter(c for _, c in train_ejemplos)
print("Distribuzione nel train:")
for cat, n in sorted(dist_train.items()):
    print(f"  {cat:<30} {n:>3} esempi")

# Mostra 2 esempi del JSONL
print("\nEsempi del file JSONL (primi 2):")
with TRAIN_FILE.open() as f:
    for i, linea in enumerate(f):
        if i >= 2: break
        ej = json.loads(linea)
        print(f"  User:      {ej['messages'][1]['content']}")
        print(f"  Assistant: {ej['messages'][2]['content']}")
        print()

## Cella 4 — Baseline: valutare il modello base senza fine-tuning

Prima di fare fine-tuning, misura quanto rende il modello base.
Questa è la tua baseline per sapere se il fine-tuning migliora qualcosa.

In [ ]:
def clasificar(titulo: str, modelo: str, few_shot_ejemplos: list[dict] | None = None) -> str:
    """Classifica un titolo. Se few_shot_ejemplos è presente, li aggiunge al contesto."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if few_shot_ejemplos:
        for ej in few_shot_ejemplos:
            messages.append({"role": "user",      "content": f"Classifica: '{ej['titulo']}'" })
            messages.append({"role": "assistant", "content": ej["categoria"]})

    messages.append({"role": "user", "content": f"Classifica: '{titulo}'"})

    resp = client.chat.completions.create(
        model=modelo,
        messages=messages,
        temperature=0.0,
        max_tokens=30,
    )
    pred = resp.choices[0].message.content.strip().lower()
    # Normalizza: estrae solo la categoria se il modello restituisce testo extra
    for cat in CATEGORIAS:
        if cat in pred:
            return cat
    return pred


def evaluar_modelo(modelo: str, eval_set: list[tuple],
                   few_shot: list[dict] | None = None,
                   max_ejemplos: int = 50) -> dict:
    """Valuta il modello sull'eval set. Restituisce accuracy e metriche per classe."""
    subset = eval_set[:max_ejemplos]
    correctas, totales = 0, 0
    por_clase = {cat: {"tp": 0, "total": 0} for cat in CATEGORIAS}

    for titulo, cat_real in subset:
        pred = clasificar(titulo, modelo, few_shot)
        correctas += pred == cat_real
        totales   += 1
        por_clase[cat_real]["total"] += 1
        if pred == cat_real:
            por_clase[cat_real]["tp"] += 1

    accuracy = correctas / totales if totales else 0
    return {
        "modelo":    modelo,
        "accuracy":  round(accuracy, 3),
        "correctas": correctas,
        "totales":   totales,
        "por_clase": {
            cat: round(v["tp"] / v["total"], 2) if v["total"] else 0
            for cat, v in por_clase.items()
        },
    }


# Valuta il modello base (senza esempi nel prompt)
print("Valutazione modello base (zero-shot)...")
resultado_base = evaluar_modelo(MODELO_BASE, eval_ejemplos, max_ejemplos=30)

print(f"\nModello base zero-shot:")
print(f"  Accuracy: {resultado_base['accuracy']:.1%}  ({resultado_base['correctas']}/{resultado_base['totales']})")
print("  Per classe:")
for cat, acc in resultado_base["por_clase"].items():
    barra = "█" * int(acc * 20) + "░" * (20 - int(acc * 20))
    print(f"    {cat:<30} {barra} {acc:.0%}")

## Cella 5 — Few-shot baseline: quanto migliora con esempi nel prompt?

In [ ]:
# Seleziona 2 esempi per categoria (10 in totale) dal train set per il few-shot
few_shot_ejemplos = []
for cat in CATEGORIAS:
    cat_train = [(t, c) for t, c in train_ejemplos if c == cat]
    for titulo, _ in cat_train[:2]:
        few_shot_ejemplos.append({"titulo": titulo, "categoria": cat})

print(f"Few-shot: {len(few_shot_ejemplos)} esempi ({len(few_shot_ejemplos)//len(CATEGORIAS)} per classe)")
print()

print("Valutazione modello base (few-shot 10 esempi)...")
resultado_fewshot = evaluar_modelo(
    MODELO_BASE, eval_ejemplos,
    few_shot=few_shot_ejemplos,
    max_ejemplos=30
)

print(f"\nModello base few-shot (10 esempi):")
print(f"  Accuracy: {resultado_fewshot['accuracy']:.1%}  ({resultado_fewshot['correctas']}/{resultado_fewshot['totales']})")
print(f"  Miglioramento vs zero-shot: +{(resultado_fewshot['accuracy']-resultado_base['accuracy'])*100:.1f}pp")

## Cella 6 — Avviare il job di fine-tuning

Carica il dataset e avvia il job. L'addestramento richiede tra 15 minuti e 1 ora
in base alla dimensione del dataset e al carico di OpenAI.

In [ ]:
JOB_CACHE = DATA_DIR / "ft_job.json"

if JOB_CACHE.exists():
    job_info = json.loads(JOB_CACHE.read_text())
    print(f"✓ Job esistente caricato dalla cache")
    print(f"  Job ID:  {job_info['job_id']}")
    print(f"  Status:  {job_info.get('status', 'sconosciuto')}")
    if job_info.get("modelo_ft"):
        print(f"  Modello: {job_info['modelo_ft']}")
else:
    # 1. Carica il file di addestramento
    print("Caricamento dataset di addestramento...")
    with TRAIN_FILE.open("rb") as f:
        upload = client.files.create(file=f, purpose="fine-tune")
    print(f"✓ File caricato: {upload.id}")

    # 2. Avvia il job
    print("Avvio job di fine-tuning...")
    job = client.fine_tuning.jobs.create(
        training_file=upload.id,
        model=MODELO_BASE,
        hyperparameters={
            "n_epochs": 3,              # più epoche = più costo, più rischio di overfitting
        },
        suffix="clasificador-pesquero",  # compare nel nome del modello
    )

    job_info = {"job_id": job.id, "file_id": upload.id, "status": job.status}
    JOB_CACHE.write_text(json.dumps(job_info, indent=2))

    print(f"✓ Job avviato: {job.id}")
    print(f"  Status iniziale: {job.status}")
    print(f"  Estimated finish: {job.estimated_finish}")

print()
print("Puoi fare polling dello stato con la cella 7.")
print("Oppure consultare: https://platform.openai.com/finetune")

## Cella 7 — Polling: attendere che il job termini

Esegui questa cella periodicamente finché lo status non è `succeeded`.
Può richiedere 15–60 minuti.

In [ ]:
job_info = json.loads(JOB_CACHE.read_text())
job_id   = job_info["job_id"]

job = client.fine_tuning.jobs.retrieve(job_id)

print(f"Job ID:  {job.id}")
print(f"Status:  {job.status}")
print(f"Modello: {job.fine_tuned_model or '(in attesa)'}")
print()

# Mostra gli ultimi eventi del job
eventos = client.fine_tuning.jobs.list_events(job_id, limit=5)
print("Ultimi eventi:")
for ev in reversed(list(eventos.data)):
    print(f"  [{ev.level}] {ev.message}")

# Salva il modello se è già pronto
if job.status == "succeeded" and job.fine_tuned_model:
    job_info["modelo_ft"] = job.fine_tuned_model
    job_info["status"]    = job.status
    JOB_CACHE.write_text(json.dumps(job_info, indent=2))
    print(f"\n✓ Modello con fine-tuning pronto: {job.fine_tuned_model}")
elif job.status == "failed":
    print(f"\n✗ Job fallito: {job.error}")
else:
    print(f"\n⏳ Job in corso — riesegui questa cella tra qualche minuto")

## Cella 8 — Valutare il modello con fine-tuning e confrontare

Esegui questa cella solo quando il job è terminato (`status: succeeded`).

In [ ]:
job_info = json.loads(JOB_CACHE.read_text())
MODELO_FT = job_info.get("modelo_ft")

if not MODELO_FT:
    print("⚠  Il modello con fine-tuning non è ancora pronto.")
    print("   Attendi che il job termini e riesegui.")
else:
    print(f"Valutazione modello con fine-tuning: {MODELO_FT}")
    print("(può richiedere 1-2 minuti)\n")

    resultado_ft = evaluar_modelo(MODELO_FT, eval_ejemplos, max_ejemplos=50)

    # ── Tabella comparativa ───────────────────────────────────────────────────
    print("=" * 65)
    print(" Confronto tra modelli")
    print("=" * 65)
    print(f"{'Modello':<35} {'Accuracy':>10} {'Corrette':>12}")
    print("-" * 65)

    resultados = [
        ("Base zero-shot",       resultado_base),
        ("Base few-shot (10es)", resultado_fewshot),
        ("Fine-tuned",           resultado_ft),
    ]
    for nombre, r in resultados:
        marca = " ←"
        print(f"  {nombre:<33} {r['accuracy']:>9.1%} {r['correctas']:>5}/{r['totales']:<5}")

    mejor = max(resultados, key=lambda x: x[1]["accuracy"])
    print(f"\n  Modello migliore: {mejor[0]} ({mejor[1]['accuracy']:.1%})")

    # ── Analisi per classe ────────────────────────────────────────────────────
    print()
    print(f"{'Categoria':<30} {'Base':>8} {'Few-shot':>10} {'Fine-tuned':>12}")
    print("-" * 65)
    for cat in CATEGORIAS:
        b  = resultado_base["por_clase"].get(cat, 0)
        fs = resultado_fewshot["por_clase"].get(cat, 0)
        ft = resultado_ft["por_clase"].get(cat, 0)
        mejor_clase = max(b, fs, ft)
        marca_b  = " ◄" if b  == mejor_clase else ""
        marca_fs = " ◄" if fs == mejor_clase else ""
        marca_ft = " ◄" if ft == mejor_clase else ""
        print(f"  {cat:<28} {b:>7.0%} {fs:>9.0%}{marca_fs:<2} {ft:>10.0%}{marca_ft}")

## Cella 9 — Analisi dei costi: vale la pena il fine-tuning?

In [ ]:
# ── Parametri del caso d'uso ──────────────────────────────────────────────────
TOKENS_POR_CLASIFICACION = 120   # system + user + risposta
CLASIFICACIONES_DIA      = 500   # documenti che arrivano al giorno
DIAS_MES                 = 22

# ── Costi di inferenza ────────────────────────────────────────────────────────
PRECIO_BASE_INPUT  = 0.15   # gpt-4o-mini, $/1M token
PRECIO_BASE_OUTPUT = 0.60
PRECIO_FT_INPUT    = 0.30   # il modello con fine-tuning costa 2× in inferenza
PRECIO_FT_OUTPUT   = 1.20

tokens_input  = int(TOKENS_POR_CLASIFICACION * 0.85)
tokens_output = int(TOKENS_POR_CLASIFICACION * 0.15)
n_mes         = CLASIFICACIONES_DIA * DIAS_MES

coste_base_mes = (tokens_input * PRECIO_BASE_INPUT + tokens_output * PRECIO_BASE_OUTPUT) / 1e6 * n_mes
coste_ft_mes   = (tokens_input * PRECIO_FT_INPUT   + tokens_output * PRECIO_FT_OUTPUT  ) / 1e6 * n_mes

# Il few-shot aggiunge token extra (~200 token per i 10 esempi)
coste_fewshot_mes = (
    (tokens_input + 200) * PRECIO_BASE_INPUT + tokens_output * PRECIO_BASE_OUTPUT
) / 1e6 * n_mes

# Costo del fine-tuning (una tantum)
tokens_train  = len(train_ejemplos) * TOKENS_POR_CLASIFICACION
coste_ft_unico = tokens_train * 3 * 3.00 / 1e6  # 3 epoche, $3/1M token

print("=" * 55)
print(f" Analisi dei costi ({CLASIFICACIONES_DIA} doc/giorno, {DIAS_MES} giorni/mese)")
print("=" * 55)
print(f"  Base zero-shot:    ${coste_base_mes:.3f}/mese")
print(f"  Base few-shot:     ${coste_fewshot_mes:.3f}/mese  (+{(coste_fewshot_mes-coste_base_mes)/coste_base_mes*100:.0f}% per token extra)")
print(f"  Fine-tuned:        ${coste_ft_mes:.3f}/mese  (2× prezzo inferenza)")
print(f"  Costo addestramento (una tantum): ${coste_ft_unico:.3f}")
print()

# Break-even: quando si ammortizza il fine-tuning rispetto al few-shot?
ahorro_mensual = coste_fewshot_mes - coste_ft_mes
if ahorro_mensual > 0:
    breakeven_meses = coste_ft_unico / ahorro_mensual
    print(f"  Il fine-tuning è più economico del few-shot a partire dal mese {breakeven_meses:.1f}")
else:
    print(f"  Il few-shot è più economico del fine-tuned a questo volume")
    print(f"  (il fine-tuning si giustifica solo se migliora sensibilmente l'accuracy)")

print()
print("Conclusione:")
if 'resultado_ft' in dir() and resultado_ft:
    mejora = resultado_ft['accuracy'] - resultado_fewshot['accuracy']
    if mejora > 0.05:
        print(f"  Il fine-tuning migliora l'accuracy di +{mejora*100:.1f}pp rispetto al few-shot → consigliato")
    elif mejora > 0:
        print(f"  Il fine-tuning migliora solo di +{mejora*100:.1f}pp → valuta se giustifica il costo extra")
    else:
        print(f"  Il few-shot supera o eguaglia il fine-tuned → usa il few-shot (più semplice)")
else:
    print("  (esegui la cella 8 per vedere la raccomandazione con dati reali)")

## Sfida — Analisi degli errori del modello con fine-tuning

L'accuracy globale non ti dice dove sbaglia il modello. Implementa un'analisi degli errori
che mostri i 5 errori più frequenti e la matrice di confusione.

```python
def analizar_errores(modelo: str, eval_set: list[tuple], max_ej: int = 50) -> dict:
    """
    Restituisce:
    - confusion_matrix: dict[real][pred] = count
    - errores: lista di {titulo, real, predicho} ordinata per frequenza dell'errore
    """
    # TODO: implementare
    # 1. Per ogni esempio, ottieni la predizione
    # 2. Se è incorretta, aggiungila alla lista degli errori
    # 3. Costruisci la matrice di confusione
    pass


def imprimir_confusion_matrix(matriz: dict):
    """Stampa la matrice di confusione in modo leggibile."""
    # TODO: implementare
    # Suggerimento: usa un formato tabellare con le categorie come righe e colonne
    pass
```

Domande su cui riflettere dopo l'analisi:
- Quale coppia di categorie si confonde di più? Ha senso semanticamente?
- Gli errori sono ambiguità reali (anche un umano avrebbe dubbi) o errori evidenti?
- Come miglioreresti il dataset per correggere gli errori più frequenti?

## Riepilogo della settimana 20

| Concetto | Dove l'hai applicato |
|---------|-------------------|
| Albero decisionale fine-tuning vs RAG vs few-shot | Dispensa + Cella 9 |
| Distillazione: GPT-4o genera il dataset | Cella 2 |
| Formato JSONL per il fine-tuning OpenAI | Cella 3 |
| Split train/eval stratificato | Cella 3 |
| Baseline zero-shot e few-shot | Celle 4-5 |
| `fine_tuning.jobs.create()` + polling | Celle 6-7 |
| Valutazione comparativa di 3 modelli | Cella 8 |
| Analisi dei costi e break-even | Cella 9 |
| Analisi degli errori + matrice di confusione | Sfida |

La settimana 21 copre **sicurezza nei sistemi LLM**: prompt injection, jailbreak,
guardrail e come proteggere un'API di produzione da usi malevoli.